# Wine Cultivar Authentication with Abstention

**Family:** End-to-end project | **Dataset:** `wine` | **Task:** classification

Classify wine cultivar from chemistry, find the smallest useful feature set, and abstain when the model is unsure.

## Table of Contents

- 1. Problem statement
- 2. Business or research context
- 3. Dataset documentation
- 4. Research references
- 5. Exploratory analysis
- 6. Preprocessing
- 7. Model selection
- 8. Model training
- 9. Evaluation
- 10. Comparison
- 11. Interpretation
- 12. Limitations
- 13. Final recommendations

## 1. Problem statement

Identify the cultivar of a wine from chemical analysis, and decide when a human should re-check.

## 2. Business or research context

Authentication checks are more useful when the system says 'I am not sure' on borderline samples. With 178 samples, estimates are noisy, so we use repeated cross validation.

## 3. Dataset documentation

**Wine**, UCI, licence CC BY 4.0, DOI 10.24432/C5PC7J. 178 rows, 13 chemical measurements (alcohol, malic acid, ash, magnesium, phenols, flavanoids, colour intensity, proline and others). Target `cultivar`, three classes. Limits: one region, tiny sample.

## 4. Research references

Verification status is from the build environment on 2026-11-17. Entries marked *not verified* must be checked before citing.

- Cover, T. and Hart, P. (1967). *Nearest neighbor pattern classification*. IEEE Transactions on Information Theory 13(1):21-27. https://doi.org/10.1109/TIT.1967.1053964 Verification: Crossref 2026-11-17
- Tibshirani, R. (1996). *Regression Shrinkage and Selection Via the Lasso*. Journal of the Royal Statistical Society B 58(1):267-288. https://doi.org/10.1111/j.2517-6161.1996.tb02080.x Verification: Crossref 2026-11-17
- Breiman, L. (2001). *Random Forests*. Machine Learning 45(1):5-32. https://doi.org/10.1023/A:1010933404324 Verification: Crossref 2026-11-17

Nearest neighbours, a lasso penalised logistic regression and a random forest are compared.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from ml_lab.data import load_dataset
from ml_lab.preprocessing import split_three_way, make_model_pipeline, build_preprocessor
from ml_lab.evaluation import evaluate_estimator, internal_scores, external_scores
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz
PROJECT_ID = "wine_authentication"
SEED = 42
set_seed(SEED)
viz.apply_style()
print_environment()

python 3.13.18 | Linux-6.18.44-fc-v595-x86_64-with-glibc2.39
  numpy==2.5.3
  pandas==3.0.5
  scipy==1.18.1
  scikit-learn==1.9.1
  matplotlib==3.11.2
  xgboost==3.4.1
  lightgbm==4.7.0
  catboost==1.2.10
  umap-learn==0.6.0
  torch==2.15.0
  gymnasium==1.4.0


In [2]:
bundle = load_dataset("wine")
X, y = bundle.X, bundle.y
print(X.shape, y.value_counts().sort_index().to_dict())

(178, 13) {0: 59, 1: 71, 2: 48}


## 5. Exploratory analysis

Class counts and the features with the largest class differences.

In [3]:
from sklearn.feature_selection import f_classif
f, _ = f_classif(X, y)
display(pd.Series(f, index=X.columns).sort_values(ascending=False).head(6).round(1).to_frame("ANOVA F"))

,ANOVA F
flavanoids,233.9
proline,207.9
od280/od315_of_diluted_wines,190.0
alcohol,135.1
color_intensity,120.7
hue,101.3


## 6. Preprocessing

Scaling inside each pipeline. With 178 rows we use repeated stratified 5-fold cross validation (5 repeats) instead of a single split.

In [4]:
from sklearn.model_selection import RepeatedStratifiedKFold
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

## 7. Model selection

Logistic regression with L1 penalty, k nearest neighbours and random forest. Selection by cross validated macro F1 on all rows is acceptable here because no test claim is made beyond the cross validated score, and nested scoring is added below.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
cands = {"logistic_l1": LogisticRegression(penalty="l1", solver="saga", C=0.5, max_iter=5000), "knn": KNeighborsClassifier(7), "random_forest": RandomForestClassifier(300, n_jobs=2, random_state=SEED)}
rows = []
for n, m in cands.items():
    s = cross_val_score(make_model_pipeline(X, m), X, y, cv=cv, scoring="f1_macro")
    rows.append({"model": n, "cv_macro_F1": s.mean(), "std": s.std(ddof=1)})
sel = pd.DataFrame(rows).set_index("model"); display(sel.round(4)); BEST = sel["cv_macro_F1"].idxmax(); print("best:", BEST)

,cv_macro_F1,std
model,,
logistic_l1,0.9773,0.0276
knn,0.9680,0.0326
random_forest,0.9792,0.0286


best: random_forest


## 8. Model training

Fit the best model on all rows for the final description. Nested scoring: the model choice is repeated inside each outer fold so the reported score is not optimistic.

In [6]:
from sklearn.model_selection import GridSearchCV
nested = []
for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y):
    best_n, best_s = None, -1
    for n, m in cands.items():
        s = cross_val_score(make_model_pipeline(X.iloc[tr], m), X.iloc[tr], y.iloc[tr], cv=3, scoring="f1_macro").mean()
        if s > best_s: best_n, best_s = n, s
    f = make_model_pipeline(X.iloc[tr], clone(cands[best_n])).fit(X.iloc[tr], y.iloc[tr])
    from sklearn.metrics import f1_score
    nested.append(f1_score(y.iloc[te], f.predict(X.iloc[te]), average="macro"))
print("nested macro F1:", round(float(np.mean(nested)), 4), "+-", round(float(np.std(nested, ddof=1)), 4))
final = make_model_pipeline(X, clone(cands[BEST])).fit(X, y)

nested macro F1: 0.9784 +- 0.0219


## 9. Evaluation

Nested score with a dummy baseline, and the accuracy and coverage trade-off when the model abstains below a confidence cut-off (cross validated probabilities).

In [7]:
from sklearn.model_selection import cross_val_predict
proba = cross_val_predict(make_model_pipeline(X, clone(cands["random_forest"])), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")
conf, pred = proba.max(axis=1), proba.argmax(axis=1)
rows = []
for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = conf >= t
    rows.append({"confidence_cutoff": t, "coverage": keep.mean(), "accuracy_on_answered": (pred[keep] == y.values[keep]).mean() if keep.any() else np.nan})
abst = pd.DataFrame(rows).set_index("confidence_cutoff"); display(abst.round(3))
base = cross_val_score(make_model_pipeline(X, DummyClassifier(strategy="prior")), X, y, cv=cv, scoring="f1_macro").mean()
METRICS = {"nested_macro_F1_mean": float(np.mean(nested)), "nested_macro_F1_std": float(np.std(nested, ddof=1)), "baseline_macro_F1": float(base),
           "accuracy_at_cutoff_0.8": float(abst.loc[0.8, "accuracy_on_answered"]), "coverage_at_cutoff_0.8": float(abst.loc[0.8, "coverage"])}
CONFIG = {"selected_model": BEST, "cv": "repeated stratified 5x5", "abstention_model": "random_forest"}
EXTRA = {"cv_table": sel.round(4).reset_index().to_dict("records"), "abstention": abst.round(4).reset_index().to_dict("records")}
ALGORITHM = BEST

record = ExperimentRecord(notebook=PROJECT_ID, family="project", algorithm=ALGORITHM, dataset="wine", task="classification", seed=SEED, config=CONFIG, metrics=METRICS, extra=EXTRA)
print("saved:", record.save())
display(pd.Series(METRICS, name="metric").round(4).to_frame())

,coverage,accuracy_on_answered
confidence_cutoff,,
0.0,1.000,0.983
0.5,0.989,0.989
0.6,0.966,0.988
0.7,0.904,0.994
0.8,0.787,1.000
0.9,0.646,1.000


saved: /home/claude/open-python/experiments/results/wine_authentication.json


,metric
nested_macro_F1_mean,0.9784
nested_macro_F1_std,0.0219
baseline_macro_F1,0.1901
accuracy_at_cutoff_0.8,1.0000
coverage_at_cutoff_0.8,0.7865


## 10. Comparison

The cross validated table above compares the models. Below, the smallest feature set that keeps most of the score, using univariate ranking inside a pipeline.

In [8]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
for k in [1, 2, 3, 5, 8, 13]:
    s = cross_val_score(make_pipeline(StandardScaler(), SelectKBest(f_classif, k=k), LogisticRegression(max_iter=2000)), X, y, cv=cv, scoring="f1_macro")
    print(f"top {k} features: macro F1 {s.mean():.3f} +- {s.std(ddof=1):.3f}")

top 1 features: macro F1 0.772 +- 0.120


top 2 features: macro F1 0.860 +- 0.067


top 3 features: macro F1 0.930 +- 0.042


top 5 features: macro F1 0.968 +- 0.039


top 8 features: macro F1 0.976 +- 0.026


top 13 features: macro F1 0.985 +- 0.017


## 11. Interpretation

L1 logistic regression keeps few features. Show which.

In [9]:
l1 = make_model_pipeline(X, cands["logistic_l1"]).fit(X, y)
coef = pd.DataFrame(l1.named_steps["model"].coef_, columns=X.columns)
print("features with a non-zero coefficient in any class:", list(coef.columns[(coef != 0).any()]))

features with a non-zero coefficient in any class:

 ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'flavanoids', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']


## 12. Limitations

178 samples from one region and three cultivars. The task is easy, so high scores say little about harder authentication problems. Confidence cut-offs come from cross validated probabilities on the same data.

## 13. Final recommendations

Use abstention to route uncertain samples to a lab check, and collect more varied samples before relying on the result.